# 🐾 Deep Learning in the Kalahari · Session 2 of 3
## *Understanding Deep Learning*, chapters 5–6: losses and training

**Transition Bio · ML Reading Group**

This series works through chapters 1–8 of Simon J.D. Prince's **_Understanding Deep Learning_** (MIT Press; free at [udlbook.github.io](https://udlbook.github.io/udlbook/)) in three sessions. Each one is about **20 min of slides and 60–70 min of hands-on puzzles**. We follow the book's notation, square brackets and all: $\mathbf{y} = \mathbf{f}[\mathbf{x}, \boldsymbol\phi]$.

| Session | Topic | Chapters |
|---|---|---|
| Session 1 | Models | Ch 1–4 |
| **Session 2** | Losses and training | Ch 5–6 |
| Session 3 | Backprop and performance | Ch 7–8 |

---

### 🦦 The story
You've just joined a long-running field station in the Kalahari that studies a wild meerkat mob. The mob is about thirty animals, run by a dominant female (in meerkat society, she's in charge). The station has a lot of data and no deep-learning expert, so you're it. Every chapter sets you a new problem from the field.

All datasets are **synthetic**. They're generated in this notebook so that we know the ground truth and can check your answers.

### 🧭 How each part works
| Symbol | Meaning |
|---|---|
| 📖 | Recap of the chapter's maths, with the equations derived properly |
| 💻 | We build it **from scratch in NumPy** first; PyTorch only appears once you've earned it |
| 🧩 ⭐ / ⭐⭐ / ⭐⭐⭐ | Puzzle: quick / medium / stretch. Fill in the `TODO`, then run the 🧪 check cell |
| 🔑 | Hidden solution cell. Double-click to open it. Have a go first; it's there when you need it |
| 🏆 | Open challenge with bronze / silver / gold targets to aim for |

> **Important:** later cells use the functions you write. If you get stuck, **run the 🔑 solution cell** (you don't have to read it) so the rest of the notebook still works.

> 📦 **Coming from Session 1?** The hidden *carry-over* cell after Setup re-creates everything this session needs from earlier parts, so you don't need to re-run old notebooks.

### 🗺️ This session (≈ 70 min hands-on)
| | | Suggested time |
|---|---|---|
| Part 5 | **Loss functions** (Ch 5): sentinels, alarm calls, scorpions and lost burrows | ~35 min |
| Part 6 | **Fitting models** (Ch 6): race to the burrow (GD, SGD, momentum, Adam) | ~35 min |

⏱ **Short on time?** Skip the stretch puzzles **5.2, 5.6, 6.3** in the room and finish them afterwards; they're good practice to finish afterwards.

At the end, `progress()` shows which puzzles you've solved and which are left. It's just for you; nothing is recorded or shared.

*Book figures and problem numbers are cited as e.g. "Fig 3.3" and "P3.8".*

In [ ]:
#@title 🐾 Setup: run me first! (plotting style, score keeper, referee)
import math, itertools, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=4, suppress=True)

# Transition Bio palette
TB = dict(green="#008759", lgreen="#73AE63", blue="#1366AB", lblue="#73B0E6",
          purple="#8F64B7", red="#CF1D10", orange="#EF8E00", grey="#6B6B6B", sand="#E9D8B4")
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (6, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=[TB["green"], TB["blue"], TB["orange"],
                                         TB["purple"], TB["red"], TB["lblue"], TB["lgreen"]]),
    "font.size": 10.5, "axes.titleweight": "bold"})

# ---------------- progress tracker (private to you; nothing is shared) ----------------
_SOLVED, _PEEKED, _SEEN = set(), set(), set()
SESSION = 2

def peek(pid):
    _PEEKED.add(pid)

def check(pid, stars, test, ok_msg="", hint=""):
    """Run `test` (a zero-arg function returning True/False) and report the result."""
    _SEEN.add(pid)
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"🕳️  Puzzle {pid}: still a TODO. Fill it in and re-run (or run the 🔑 cell).")
        return
    except Exception as e:
        print(f"💥 Puzzle {pid} raised {type(e).__name__}: {e}")
        if hint: print("   💡 Hint:", hint)
        return
    if ok:
        _SOLVED.add(pid)
        print(f"✅ Puzzle {pid} solved!  {ok_msg}")
    else:
        print(f"❌ Puzzle {pid}: not quite.", ("💡 " + hint) if hint else "")

def progress():
    own = sorted(_SOLVED - _PEEKED); helped = sorted(_SOLVED & _PEEKED); todo = sorted(_SEEN - _SOLVED)
    print(f"🐾 {len(_SOLVED)} of {len(_SEEN)} puzzles solved "
          f"({len(own)} on your own, {len(helped)} after opening the solution).")
    if todo: print("   Still to do:", ", ".join(todo))

def close(a, b, tol=1e-6):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def num_grad(fn, params, eps=1e-6):
    """Central finite-difference gradient of scalar fn(params) w.r.t. a 1D array."""
    params = np.array(params, float); g = np.zeros_like(params)
    for i in range(len(params)):
        d = np.zeros_like(params); d[i] = eps
        g[i] = (fn(params + d) - fn(params - d)) / (2 * eps)
    return g

def medal(score, bronze, silver, gold, higher_is_better=False):
    s = score if higher_is_better else -score
    b, si, g = (bronze, silver, gold) if higher_is_better else (-bronze, -silver, -gold)
    if s >= g: return "🥇 GOLD"
    if s >= si: return "🥈 SILVER"
    if s >= b: return "🥉 BRONZE"
    return "🐣 no medal yet: keep digging"

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
print("Setup complete 🐾  | numpy", np.__version__, "| torch available:", HAS_TORCH)

In [ ]:
#@title 📦 Carry-over from Session 1 (run me: re-creates what Parts 5–6 need)
def relu(z):
    return np.maximum(z, 0.0)

# Part 2: the pup weigh-in data and the linear model / least squares loss
rng = np.random.default_rng(2)
I = 20
hours = np.sort(rng.uniform(0.5, 7.5, I))
gain = 1.5 + 2.2 * hours + rng.normal(0, 2.0, I)
def f_linear(x, phi0, phi1):
    return phi0 + phi1 * x
def loss_ls(phi0, phi1, x, y):
    return np.sum((f_linear(x, phi0, phi1) - y) ** 2)
print("Carried over: relu, hours, gain, f_linear, loss_ls 🐾")

---
# Part 5 · Loss functions (Chapter 5): 🔭 sentinels, alarm calls, scorpions and lost burrows
### 📖 The recipe (§5.2)
Instead of predicting $\mathbf{y}$ directly, the network predicts the **parameters $\boldsymbol\theta$ of a distribution** over $\mathbf{y}$.
1. Choose a distribution $Pr(\mathbf{y}\,|\,\boldsymbol\theta)$ defined on the output domain.
2. Let the network predict the parameters: $\boldsymbol\theta=\mathbf{f}[\mathbf{x},\boldsymbol\phi]$.
3. Train by **maximum likelihood**. Because $\log$ is monotonic and we prefer minimizing, minimize the **negative log-likelihood**:
$$\hat{\boldsymbol\phi}=\underset{\boldsymbol\phi}{\mathrm{argmin}}\Big[-\sum_{i=1}^{I}\log Pr\big(\mathbf{y}_i\,|\,\mathbf{f}[\mathbf{x}_i,\boldsymbol\phi]\big)\Big].$$
4. **Inference**: return the full distribution, or its maximum $\hat{\mathbf{y}}=\mathrm{argmax}_{\mathbf{y}}\,Pr(\mathbf{y}\,|\,\mathbf{f}[\mathbf{x},\hat{\boldsymbol\phi}])$.

| output domain | distribution | final activation | loss |
|---|---|---|---|
| $y\in\mathbb{R}$ | normal | none | least squares |
| $y\in\{0,1\}$ | Bernoulli | logistic sigmoid | binary cross-entropy |
| $y\in\{1..K\}$ | categorical | softmax | multiclass cross-entropy |
| $y\in\{0,1,2,\dots\}$ | Poisson | $\exp$ | Poisson NLL |
| $y\in(-\pi,\pi]$ | von Mises | none | $-\kappa\cos(y-\mu)$ |

## 5A · Univariate regression: least squares is a *derived* result
With $Pr(y|\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\big[-\frac{(y-\mu)^2}{2\sigma^2}\big]$ and $\mu=\mathrm{f}[x,\boldsymbol\phi]$:
$$L[\boldsymbol\phi]=-\sum_i\log Pr(y_i|\mathrm{f}[x_i,\boldsymbol\phi],\sigma^2)=\frac{I}{2}\log(2\pi\sigma^2)+\sum_i\frac{(y_i-\mathrm{f}[x_i,\boldsymbol\phi])^2}{2\sigma^2}.$$
The first term doesn't depend on $\boldsymbol\phi$, and $2\sigma^2$ is a positive constant, so the argmin is exactly the **least squares** solution (eq 5.11). The Gaussian assumption is what justifies the loss we used in Part 2.

### 🧩 Puzzle 5.1 ⭐⭐ Normal NLL and the MLE of the variance (§5.3.3)
1. Implement `normal_nll(y, mu, sigma2)` (a sum over points).
2. Differentiate $L$ with respect to $\sigma^2$, set the result to zero, and implement `sigma2_mle(y, mu)`.

In [ ]:
def normal_nll(y, mu, sigma2):
    # TODO
    raise NotImplementedError

def sigma2_mle(y, mu):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.1
def _t51():
    rr = np.random.default_rng(0); y = rr.normal(3, 2, 200); mu = np.full(200, 3.1)
    ref = np.sum(0.5 * np.log(2 * np.pi * 1.7) + (y - mu) ** 2 / (2 * 1.7))
    grid = np.linspace(0.5, 10, 20001); best = grid[np.argmin([normal_nll(y, mu, s) for s in grid])]
    phis = np.linspace(-1, 6, 701)
    same_argmin = all(phis[np.argmin([normal_nll(gain, f_linear(hours, 1.0, p), s2) for p in phis])] ==
                      phis[np.argmin([loss_ls(1.0, p, hours, gain) for p in phis])] for s2 in [0.3, 4, 50])
    return np.isclose(normal_nll(y, mu, 1.7), ref) and abs(sigma2_mle(y, mu) - best) < 1e-3 and same_argmin
check("5.1", 2, _t51, "sigma^2_hat = mean squared residual. Changing sigma never moves the best phi.",
      "dL/dsigma2 = I/(2 sigma2) - sum r^2 / (2 sigma2^2) = 0")

In [ ]:
#@title 🔑 Solution 5.1 — double-click to peek (have a go first!)
peek('5.1')
def normal_nll(y, mu, sigma2):
    return np.sum(0.5 * np.log(2 * np.pi * sigma2) + (y - mu) ** 2 / (2 * sigma2))
def sigma2_mle(y, mu):
    return np.mean((y - mu) ** 2)

### 📖 Heteroscedastic regression (§5.3.4): pups get *more variable* as they grow
Give the network **two outputs**: a mean $\mu=\mathrm{f}_1[x,\boldsymbol\phi]$ and a variance $\sigma^2=\mathrm{softplus}[\mathrm{f}_2[x,\boldsymbol\phi]]$, which keeps the variance positive. Minimize
$$L=\sum_i\Big[\tfrac12\log(2\pi\sigma_i^2)+\frac{(y_i-\mu_i)^2}{2\sigma_i^2}\Big].$$

### 🧩 Puzzle 5.2 ⭐⭐ Heteroscedastic NLL in PyTorch
Write the loss as a torch function, averaged over the batch. A small network then learns how pup mass varies with age, including the growing spread.

In [ ]:
def hetero_nll(y, out):
    """y: (N,), out: (N, 2) raw network outputs [mu, raw_var]. sigma2 = softplus(raw_var) + 1e-6.
    Return the MEAN negative log-likelihood (a scalar tensor)."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.2
def _t52():
    if not HAS_TORCH: print("   (torch missing, skipping)"); return True
    out = torch.tensor([[1.0, 0.0], [2.0, -1.0], [0.5, 3.0]]); y = torch.tensor([1.5, 1.0, -0.5])
    s2 = torch.nn.functional.softplus(out[:, 1]) + 1e-6
    ref = (0.5 * torch.log(2 * math.pi * s2) + (y - out[:, 0]) ** 2 / (2 * s2)).mean()
    return torch.allclose(hetero_nll(y, out), ref)
check("5.2", 2, _t52, "Now the network can say how sure it is.",
      "torch.nn.functional.softplus(out[:,1]) + 1e-6")

In [ ]:
#@title 🔑 Solution 5.2 — double-click to peek (have a go first!)
peek('5.2')
def hetero_nll(y, out):
    mu = out[:, 0]; s2 = torch.nn.functional.softplus(out[:, 1]) + 1e-6
    return (0.5 * torch.log(2 * math.pi * s2) + (y - mu) ** 2 / (2 * s2)).mean()

In [ ]:
# 💻 Train the heteroscedastic pup-growth model
if HAS_TORCH:
    torch.manual_seed(0); rr = np.random.default_rng(4)
    age = rr.uniform(0, 1, 400)                                     # age (scaled 0..1 ≈ 0..120 days)
    mass = 30 + 220 * (1 - np.exp(-2.5 * age)) + rr.normal(0, 4 + 30 * age)   # noise grows with age
    X = torch.tensor(age, dtype=torch.float32)[:, None]; Y = torch.tensor((mass - 150) / 50, dtype=torch.float32)
    net = torch.nn.Sequential(torch.nn.Linear(1, 32), torch.nn.ReLU(), torch.nn.Linear(32, 32),
                              torch.nn.ReLU(), torch.nn.Linear(32, 2))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for step in range(2000):
        opt.zero_grad(); loss = hetero_nll(Y, net(X)); loss.backward(); opt.step()
    with torch.no_grad():
        g = torch.linspace(0, 1, 200)[:, None]; o = net(g)
        mu = o[:, 0].numpy() * 50 + 150; sd = np.sqrt(torch.nn.functional.softplus(o[:, 1]).numpy() + 1e-6) * 50
    plt.scatter(age * 120, mass, s=6, color=TB["orange"], alpha=0.6, label="pups")
    plt.plot(g[:, 0] * 120, mu, color=TB["green"], lw=2, label=r"$\mu$")
    plt.fill_between(g[:, 0].numpy() * 120, mu - 2 * sd, mu + 2 * sd, color=TB["lgreen"], alpha=0.3, label=r"$\mu\pm2\sigma$")
    plt.xlabel("age (days)"); plt.ylabel("mass (g)"); plt.legend(); plt.title("Heteroscedastic regression"); plt.show()

## 5B · Binary classification: 🔭 will this meerkat go on sentinel duty?
Meerkats tend to stand guard once they've **eaten well**: a full stomach makes guarding cheap. Our feature is $x$ = grams of food eaten in the last hour, and the label is $y=1$ if the animal goes on guard.

- **Bernoulli**: $Pr(y|\lambda)=(1-\lambda)^{1-y}\lambda^{y}$, with $\lambda=\mathrm{sig}[\mathrm{f}[x,\boldsymbol\phi]]$ and $\mathrm{sig}[z]=\frac{1}{1+e^{-z}}$.
- **Binary cross-entropy** (eq 5.20): $L=\sum_i -(1-y_i)\log\big(1-\mathrm{sig}[\mathrm{f}_i]\big)-y_i\log\mathrm{sig}[\mathrm{f}_i]$.

### 🧩 Puzzle 5.3 ⭐⭐ The prettiest derivative in deep learning
Show that $\dfrac{\partial \ell_i}{\partial \mathrm{f}_i}=\mathrm{sig}[\mathrm{f}_i]-y_i$. (Use $\mathrm{sig}'[z]=\mathrm{sig}[z](1-\mathrm{sig}[z])$.) Then implement both functions below. The loss must be **numerically stable**: no `log(0)` for large $|f|$. Hint: $-\log\mathrm{sig}[f]=\mathrm{softplus}[-f]$, and `np.logaddexp(0, z)` computes $\log(1+e^{z})$ safely.

In [ ]:
def bce_from_logits(f, y):
    """Sum of binary cross-entropy terms, computed stably from logits f."""
    # TODO
    raise NotImplementedError

def bce_grad_logits(f, y):
    """d(loss)/df, elementwise."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.3
def _t53():
    f = np.array([-3.0, 0.2, 4.0, 40.0, -40.0]); y = np.array([0, 1, 1, 0, 1.0])
    sg = 1 / (1 + np.exp(-f[:3]))
    ref3 = np.sum(-(1 - y[:3]) * np.log(1 - sg) - y[:3] * np.log(sg))
    stable = np.isfinite(bce_from_logits(f, y)) and np.isclose(bce_from_logits(f[3:], y[3:]), 80, atol=1e-6)
    g_num = num_grad(lambda q: bce_from_logits(q, y), f)
    return np.isclose(bce_from_logits(f[:3], y[:3]), ref3) and stable and close(bce_grad_logits(f, y), g_num, 1e-5)
check("5.3", 2, _t53, "sig(f) - y: the error signal is just 'prediction minus truth'.",
      "loss = sum(y*logaddexp(0,-f) + (1-y)*logaddexp(0,f)); grad = 1/(1+exp(-f)) - y")

In [ ]:
#@title 🔑 Solution 5.3 — double-click to peek (have a go first!)
peek('5.3')
def bce_from_logits(f, y):
    return np.sum(y * np.logaddexp(0, -f) + (1 - y) * np.logaddexp(0, f))
def bce_grad_logits(f, y):
    return 1 / (1 + np.exp(-f)) - y

In [ ]:
# 💻 Plot the loss vs predicted probability (book P5.2), then fit logistic regression from scratch
lam = np.linspace(1e-3, 1 - 1e-3, 300)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(lam, -np.log(1 - lam), label="y = 0", color=TB["blue"]); ax[0].plot(lam, -np.log(lam), label="y = 1", color=TB["orange"])
ax[0].set_xlabel(r"$\mathrm{sig}[f]$"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Confidently wrong = very expensive")

rr = np.random.default_rng(7); food = rr.uniform(0, 30, 120)
guard = (rr.uniform(size=120) < 1 / (1 + np.exp(-(food - 15) / 3))).astype(float)
phi = np.zeros(2)
for _ in range(5000):
    fz = phi[0] + phi[1] * (food - 15) / 10
    gz = bce_grad_logits(fz, guard)
    phi -= 0.01 * np.array([gz.sum(), (gz * (food - 15) / 10).sum()])
xs = np.linspace(0, 30, 200)
ax[1].scatter(food, guard + rr.normal(0, 0.02, 120), s=10, color=TB["grey"], label="observed")
ax[1].plot(xs, 1 / (1 + np.exp(-(phi[0] + phi[1] * (xs - 15) / 10))), color=TB["green"], lw=2, label="Pr(guard | food)")
ax[1].set_xlabel("food eaten in last hour (g)"); ax[1].legend(); ax[1].set_title("Selfish sentinels"); plt.tight_layout(); plt.show()

## 5C · Multiclass classification: 🦅🐺🐍 decoding alarm calls
Meerkats give **different alarm calls for different predators**. Two acoustic features (peak frequency and call duration) should let us classify the call as **eagle / jackal / cobra**.

- **Categorical** distribution with $Pr(y=k|\mathbf{x})=\mathrm{softmax}_k[\mathbf{f}[\mathbf{x},\boldsymbol\phi]]=\dfrac{\exp[\mathrm{f}_k]}{\sum_{k'}\exp[\mathrm{f}_{k'}]}$ (eq 5.22).
- **Multiclass cross-entropy** (eq 5.24): $L=-\sum_i\Big(\mathrm{f}_{y_i}[\mathbf{x}_i,\boldsymbol\phi]-\log\sum_k\exp\big[\mathrm{f}_k[\mathbf{x}_i,\boldsymbol\phi]\big]\Big)$.
- Its gradient with respect to the logits is $\mathrm{softmax}[\mathbf{f}]-\mathbf{e}_{y}$, where $\mathbf{e}_y$ is the one-hot label. It's the same "prediction minus truth" as in 5B.

### 🧩 Puzzle 5.4 ⭐ A softmax that never overflows
`np.exp(1000)` is `inf`. Use the fact that $\mathrm{softmax}[\mathbf{z}]=\mathrm{softmax}[\mathbf{z}-c]$ for any constant $c$. Columns are examples, so normalise over **axis 0**.

In [ ]:
def softmax(z):
    """z: (K, N) logits -> (K, N) probabilities, normalised over axis 0, stable for huge z."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.4
def _t54():
    z = np.array([[1000., 1.], [1001., 2.], [1002., 3.]])
    p = softmax(z); ref = np.exp(z[:, 1]) / np.exp(z[:, 1]).sum()
    return np.all(np.isfinite(p)) and close(p[:, 0], ref) and close(p[:, 1], ref) and close(p.sum(0), [1, 1])
check("5.4", 1, _t54, "Subtracting the max: the oldest trick in numerical deep learning.",
      "z = z - z.max(axis=0, keepdims=True)")

In [ ]:
#@title 🔑 Solution 5.4 — double-click to peek (have a go first!)
peek('5.4')
def softmax(z):
    e = np.exp(z - z.max(axis=0, keepdims=True))
    return e / e.sum(axis=0, keepdims=True)

In [ ]:
# 💻 Linear softmax classifier from scratch: y = softmax(beta + Omega x), trained by gradient descent
rr = np.random.default_rng(11)
centres = np.array([[4.2, 0.20], [2.0, 0.55], [3.1, 1.10]])      # (kHz, s)  eagle / jackal / cobra
names = ["eagle", "jackal", "cobra"]; cols = [TB["blue"], TB["orange"], TB["green"]]
Xc = np.concatenate([rr.normal(c, [0.45, 0.15], (80, 2)) for c in centres]).T          # (2, 240)
yc = np.repeat([0, 1, 2], 80); mu_, sd_ = Xc.mean(1, keepdims=True), Xc.std(1, keepdims=True)
Xn = (Xc - mu_) / sd_; onehot = np.eye(3)[yc].T
Om, be = np.zeros((3, 2)), np.zeros((3, 1)); losses = []
for it in range(600):
    P = softmax(be + Om @ Xn); G = P - onehot                         # dL/df  (3, N)
    Om -= 0.5 * G @ Xn.T / Xn.shape[1]; be -= 0.5 * G.mean(1, keepdims=True)
    losses.append(-np.mean(np.log(P[yc, np.arange(len(yc))])))
g1, g2 = np.meshgrid(np.linspace(0.5, 5.8, 300), np.linspace(-0.3, 1.8, 300))
G = (np.stack([g1.ravel(), g2.ravel()]) - mu_) / sd_; pred = softmax(be + Om @ G).argmax(0).reshape(g1.shape)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(g1, g2, pred, levels=[-.5, .5, 1.5, 2.5], colors=cols, alpha=0.18)
for k in range(3): ax[0].scatter(*Xc[:, yc == k], s=10, color=cols[k], label=names[k])
ax[0].set_xlabel("peak frequency (kHz)"); ax[0].set_ylabel("duration (s)"); ax[0].legend(); ax[0].set_title("Alarm-call decoder")
ax[1].plot(losses, color=TB["green"]); ax[1].set_xlabel("iteration"); ax[1].set_ylabel("mean cross-entropy"); plt.show()
print("training accuracy:", np.mean(softmax(be + Om @ Xn).argmax(0) == yc))

## 5D · Counts: 🦂 scorpions per hour (book P5.6)
Meerkats eat scorpions (they're largely immune to the venom). The number caught per hour, $y\in\{0,1,2,\dots\}$, rises with temperature. Use a **Poisson** distribution $Pr(y=k)=\frac{\lambda^ke^{-\lambda}}{k!}$, with the rate predicted through an $\exp$ so that it stays positive: $\lambda=\exp[\mathrm{f}[x,\boldsymbol\phi]]$.

### 🧩 Puzzle 5.5 ⭐⭐ Derive and fit the Poisson loss
1. Show that $L=\sum_i\big[\exp(\mathrm{f}_i)-y_i\mathrm{f}_i+\log(y_i!)\big]$ and $\partial\ell_i/\partial\mathrm{f}_i=\exp(\mathrm{f}_i)-y_i$ (prediction minus truth again!).
2. Implement both, then fit $\mathrm{f}=\phi_0+\phi_1 x$ by gradient descent, with $x=(T-25)/10$ the scaled temperature. The true values are $\phi=[1.2, 0.6]$; can you recover them?

In [ ]:
def poisson_nll(y, f):
    # TODO  (math.lgamma(k+1) = log(k!))
    raise NotImplementedError

def poisson_grad_f(y, f):
    # TODO
    raise NotImplementedError

def fit_poisson(x, y, lr=1e-3, n_steps=5000):
    """Gradient descent on [phi0, phi1]; return np.array([phi0, phi1])."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.5
rr = np.random.default_rng(21); temp = rr.uniform(12, 40, 300); xt = (temp - 25) / 10
scorpions = rr.poisson(np.exp(1.2 + 0.6 * xt))
def _t55():
    f = np.array([0.1, 1.0, 2.0]); y = np.array([0, 3, 7])
    ref = np.sum(np.exp(f) - y * f + np.array([math.lgamma(k + 1) for k in y]))
    ok1 = np.isclose(poisson_nll(y, f), ref) and close(poisson_grad_f(y, f), num_grad(lambda q: poisson_nll(y, q), f), 1e-5)
    phi = fit_poisson(xt, scorpions); print("   recovered phi:", np.round(phi, 3))
    return ok1 and np.all(np.abs(phi - [1.2, 0.6]) < 0.08)
check("5.5", 2, _t55, "Canonical link + matching loss always gives (prediction - truth).",
      "grad wrt phi = sum((exp(f)-y) * [1, x])")

In [ ]:
#@title 🔑 Solution 5.5 — double-click to peek (have a go first!)
peek('5.5')
def poisson_nll(y, f):
    return np.sum(np.exp(f) - y * f + np.array([math.lgamma(k + 1) for k in np.atleast_1d(y)]))
def poisson_grad_f(y, f):
    return np.exp(f) - y
def fit_poisson(x, y, lr=1e-3, n_steps=5000):
    phi = np.zeros(2)
    for _ in range(n_steps):
        g = poisson_grad_f(y, phi[0] + phi[1] * x)
        phi -= lr * np.array([g.sum(), (g * x).sum()])
    return phi

## 5E · Directions: 🧭 which way is the burrow? (book P5.3)
At dusk each foraging group heads home. The heading $y\in(-\pi,\pi]$ is an **angle**, and angles wrap around: $+179°$ and $-179°$ are only 2° apart. The **von Mises** distribution handles this:
$$Pr(y|\mu,\kappa)=\frac{\exp[\kappa\cos(y-\mu)]}{2\pi\,\mathrm{Bessel}_0[\kappa]}\quad\Rightarrow\quad L=-\sum_i\kappa\cos(y_i-\mu_i)+\text{const}.$$

### 🧩 Puzzle 5.6 ⭐⭐⭐ The naive average sends the mob the wrong way
For a constant model $\mu$, set $\partial L/\partial\mu=0$ and show that $\sum_i\sin(y_i-\mu)=0$, which gives $\hat\mu=\mathrm{atan2}\big(\sum_i\sin y_i,\ \sum_i\cos y_i\big)$. Implement it, and compare with `np.mean` on headings that cluster around due **west** ($\pm\pi$).

In [ ]:
def vonmises_mu_mle(angles):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.6
def _t56():
    rr = np.random.default_rng(3)
    heads = np.angle(np.exp(1j * (np.pi + rr.normal(0, 0.3, 50))))      # around ±pi, wrapped
    mu = vonmises_mu_mle(heads); naive = heads.mean()
    print(f"   naive mean = {np.degrees(naive):7.1f}°   von Mises MLE = {np.degrees(mu):7.1f}°   (truth: ±180°)")
    grid = np.linspace(-np.pi, np.pi, 100001); Lg = [-np.sum(np.cos(heads - m)) for m in grid[::100]]
    m_best = grid[::100][np.argmin(Lg)]
    return abs(np.angle(np.exp(1j * (mu - m_best)))) < 0.01
check("5.6", 3, _t56, "Least squares on angles would send the mob off due EAST.",
      "np.arctan2(np.sin(a).sum(), np.cos(a).sum())")

In [ ]:
#@title 🔑 Solution 5.6 — double-click to peek (have a go first!)
peek('5.6')
def vonmises_mu_mle(angles):
    return np.arctan2(np.sin(angles).sum(), np.cos(angles).sum())

### 📖 Cross-entropy, KL divergence and why it's all the same thing (§5.7)
Let $q(y)$ be the empirical distribution of the data, a set of spikes at the $y_i$. Minimizing the KL divergence $D_{KL}[q\,\|\,Pr(y|\boldsymbol\theta)]=\int q\log q\,dy-\int q\log Pr(y|\boldsymbol\theta)\,dy$ over $\boldsymbol\theta$ drops the first term, since it doesn't depend on $\boldsymbol\theta$. What's left is the **cross-entropy** $-\frac1I\sum_i\log Pr(y_i|\boldsymbol\theta)$: our negative log-likelihood. So "cross-entropy loss" and "maximum likelihood" are two names for the same objective.

---
# Part 6 · Fitting models (Chapter 6): 🏃 race to the burrow
### 📖 Recap
**Gradient descent** (eq 6.3): $\boldsymbol\phi\leftarrow\boldsymbol\phi-\alpha\,\dfrac{\partial L}{\partial\boldsymbol\phi}$, where $\alpha$ is the learning rate.

Linear-regression losses are **convex**: every chord lies above the surface, so there's one minimum. Neural network losses are **non-convex**, with local minima and saddle points. The book explores this with the 2-parameter **Gabor model** (eq 6.8):
$$\mathrm{f}[x,\boldsymbol\phi]=\sin[\phi_0+0.06\,\phi_1x]\cdot\exp\!\Big(-\frac{(\phi_0+0.06\,\phi_1x)^2}{32}\Big).$$

**SGD** (eq 6.10) takes each step using a random **minibatch** $\mathcal{B}_t$: $\;\boldsymbol\phi_{t+1}\leftarrow\boldsymbol\phi_t-\alpha\sum_{i\in\mathcal{B}_t}\partial\ell_i/\partial\boldsymbol\phi$. One pass through the whole dataset is an **epoch**.

**Momentum** (eq 6.11): $\;\mathbf{m}_{t+1}\leftarrow\beta\mathbf{m}_t+(1-\beta)\sum_{i\in\mathcal{B}_t}\partial\ell_i/\partial\boldsymbol\phi,\quad \boldsymbol\phi_{t+1}\leftarrow\boldsymbol\phi_t-\alpha\,\mathbf{m}_{t+1}$.
**Nesterov** (eq 6.12) evaluates the gradient at the look-ahead point $\boldsymbol\phi_t-\alpha\beta\,\mathbf{m}_t$ instead.

**Adam** (eqs 6.15–6.17):
$$\mathbf{m}_{t+1}\leftarrow\beta\mathbf{m}_t+(1-\beta)\mathbf{g}_t,\quad \mathbf{v}_{t+1}\leftarrow\gamma\mathbf{v}_t+(1-\gamma)\mathbf{g}_t^2,$$
$$\tilde{\mathbf{m}}_{t+1}=\frac{\mathbf{m}_{t+1}}{1-\beta^{t+1}},\quad\tilde{\mathbf{v}}_{t+1}=\frac{\mathbf{v}_{t+1}}{1-\gamma^{t+1}},\quad \boldsymbol\phi_{t+1}\leftarrow\boldsymbol\phi_t-\alpha\frac{\tilde{\mathbf{m}}_{t+1}}{\sqrt{\tilde{\mathbf{v}}_{t+1}}+\epsilon}.$$
The bias correction fixes the too-small early estimates (since $\mathbf{m}_0=\mathbf{v}_0=\mathbf{0}$). Dividing by $\sqrt{\tilde{\mathbf v}}$ normalises the step size in every direction.

In [ ]:
# 💻 The Gabor model and its dataset (28 points, true phi = [0, 16.6], as in Fig 6.3)
def gabor(x, phi):
    u = phi[0] + 0.06 * phi[1] * x
    return np.sin(u) * np.exp(-u ** 2 / 32.0)

rr = np.random.default_rng(8)
xg = np.sort(rr.uniform(-15, 15, 28)); yg = gabor(xg, [0.0, 16.6]) + rr.normal(0, 0.1, 28)
def gabor_loss(phi, x=xg, y=yg):
    return np.sum((gabor(x, phi) - y) ** 2)

A0, A1 = np.meshgrid(np.linspace(-10, 10, 241), np.linspace(2.5, 22.5, 241))
LG = np.array([[gabor_loss([a, b]) for a, b in zip(r0, r1)] for r0, r1 in zip(A0, A1)])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(xg, yg, color=TB["orange"]); xx = np.linspace(-15, 15, 400)
ax[0].plot(xx, gabor(xx, [0, 16.6]), color=TB["green"]); ax[0].set_title("Gabor data")
c = ax[1].contourf(A0, A1, LG, 30, cmap="viridis"); plt.colorbar(c, ax=ax[1])
ax[1].set_xlabel(r"$\phi_0$"); ax[1].set_ylabel(r"$\phi_1$"); ax[1].set_title("Non-convex loss: many valleys"); plt.show()

### 🧩 Puzzle 6.1 ⭐⭐ Gradient of the Gabor loss (book P6.3)
Let $u=\phi_0+0.06\phi_1x$ and $g(u)=\sin u\cdot e^{-u^2/32}$. Then
$$g'(u)=\cos u\; e^{-u^2/32}-\frac{u}{16}\sin u\; e^{-u^2/32},\qquad \frac{\partial \mathrm{f}}{\partial\phi_0}=g'(u),\qquad \frac{\partial \mathrm{f}}{\partial\phi_1}=0.06\,x\,g'(u).$$
Use these and the chain rule to implement `gabor_grad`.

In [ ]:
def gabor_grad(phi, x=xg, y=yg):
    """Return np.array([dL/dphi0, dL/dphi1]) for the least squares Gabor loss."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 6.1
def _t61():
    return all(close(gabor_grad(np.array(p)), num_grad(gabor_loss, p), 1e-4) for p in [(0.3, 15.), (-4, 8.), (6, 20.)])
check("6.1", 2, _t61, "That's analytic gradients. Part 7 automates this for any network.",
      "dL/dphi = sum 2*(f - y) * df/dphi")

In [ ]:
#@title 🔑 Solution 6.1 — double-click to peek (have a go first!)
peek('6.1')
def gabor_grad(phi, x=xg, y=yg):
    u = phi[0] + 0.06 * phi[1] * x; e = np.exp(-u ** 2 / 32.0)
    dg = np.cos(u) * e - (u / 16.0) * np.sin(u) * e
    r = 2 * (np.sin(u) * e - y)
    return np.array([np.sum(r * dg), np.sum(r * dg * 0.06 * x)])

In [ ]:
# 💻 Full-batch GD vs SGD from the same starting points (Fig 6.5)
def gd(grad_fn, phi, lr, n):
    path = [np.array(phi, float)]
    for _ in range(n): path.append(path[-1] - lr * grad_fn(path[-1]))
    return np.array(path)

def sgd_gabor(phi, lr, n, batch=7, seed=0):
    r = np.random.default_rng(seed); path = [np.array(phi, float)]; order = r.permutation(28); pos = 0
    for t in range(n):
        lr_t = lr * (0.5 ** (t // 600))                        # learning-rate schedule: halve every 600 steps
        if pos + batch > 28: order, pos = r.permutation(28), 0  # new epoch
        idx = order[pos:pos + batch]; pos += batch
        path.append(path[-1] - lr_t * gabor_grad(path[-1], xg[idx], yg[idx]))
    return np.array(path)

starts = [(-6, 20), (3, 15), (2, 19), (1, 12)]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, title in zip(ax, ["Gradient descent", "Stochastic gradient descent (batch 7)"]):
    a.contourf(A0, A1, LG, 30, cmap="viridis"); a.set_title(title); a.set_xlabel(r"$\phi_0$")
for s, col in zip(starts, [TB["orange"], "w", TB["red"], TB["lblue"]]):
    p = gd(gabor_grad, s, 0.01, 2000); ax[0].plot(p[:, 0], p[:, 1], ".-", color=col, ms=2, lw=1)
    q = sgd_gabor(s, 0.2, 2000, seed=3); ax[1].plot(q[:, 0], q[:, 1], ".-", color=col, ms=2, lw=0.8)
    print(f"start {s}: GD ends at loss {gabor_loss(p[-1]):6.3f}   SGD ends at loss {gabor_loss(q[-1]):6.3f}")
for a in ax: a.plot(0, 16.6, "*", color="w", ms=14)
ax[0].set_ylabel(r"$\phi_1$"); plt.show()
esc = sum(gabor_loss(sgd_gabor((3, 15), 0.2, 2000, seed=k)[-1]) < 0.5 for k in range(20))
print(f"From (3, 15) GD always gets stuck. SGD reached the global minimum (loss ≈ 0.44) in {esc}/20 random runs.")
print("Lesson: in a 2-parameter landscape the starting point dominates; SGD *can* escape, but there's no guarantee.")

### 🧩 Puzzle 6.2 ⭐ Quick-fire (book P6.9 & P6.11)
1. We run SGD for **1,000 iterations** on a dataset of **100** examples with **batch size 20**. How many epochs is that?
2. A model has **one million** parameters. How many **entries** does its Hessian have, and how many **terabytes** would it take in float32 (4 bytes each; 1 TB = $10^{12}$ bytes)?

In [ ]:
epochs = None          # TODO
hessian_entries = None # TODO
hessian_TB = None      # TODO

In [ ]:
# 🧪 Check puzzle 6.2
def _t62():
    if None in (epochs, hessian_entries, hessian_TB): raise NotImplementedError
    return epochs == 200 and hessian_entries == 10 ** 12 and np.isclose(hessian_TB, 4)
check("6.2", 1, _t62, "This is why deep learning uses first-order optimisers: no Hessians allowed.",
      "batches per epoch = 100/20")

In [ ]:
#@title 🔑 Solution 6.2 — double-click to peek (have a go first!)
peek('6.2')
epochs = 1000 // (100 // 20)
hessian_entries = (10 ** 6) ** 2
hessian_TB = hessian_entries * 4 / 1e12

### 🧩 Puzzle 6.3 ⭐⭐ Momentum is a weighted average of past gradients (book P6.10)
Unroll the recursion $\mathbf{m}_{t+1}=\beta\mathbf{m}_t+(1-\beta)\mathbf{g}_t$, starting from $\mathbf{m}_0=\mathbf{0}$. The gradient from $k$ steps ago has weight $w_k=\;?$

Implement `momentum_weight(k, beta)`. The checker confirms it against a simulation and checks that the weights sum to 1 as $t\to\infty$.

In [ ]:
def momentum_weight(k, beta):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 6.3
def _t63():
    beta, T = 0.9, 60; G = np.eye(T); m = np.zeros(T)
    for t in range(T): m = beta * m + (1 - beta) * G[t]      # m after T steps, as a function of each g_t
    sim = m[::-1]                                            # weight of gradient k steps ago
    return close([momentum_weight(k, beta) for k in range(T)], sim, 1e-12) and np.isclose(sum(momentum_weight(k, beta) for k in range(2000)), 1)
check("6.3", 2, _t63, "An exponential moving average with an effective memory of ~1/(1-beta) steps.",
      "w_k = (1 - beta) * beta**k")

In [ ]:
#@title 🔑 Solution 6.3 — double-click to peek (have a go first!)
peek('6.3')
def momentum_weight(k, beta):
    return (1 - beta) * beta ** k

### 🧩 Puzzle 6.4 ⭐⭐ Implement momentum, Nesterov and Adam from the equations
Each optimiser takes `grad_fn(phi) -> gradient` and returns the **path**: an array of shape `(n_steps+1, n_params)` that includes the start. The checker compares your paths with reference implementations.

In [ ]:
def momentum(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    # TODO
    raise NotImplementedError

def nesterov(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    # TODO  (gradient evaluated at phi - alpha*beta*m)
    raise NotImplementedError

def adam(grad_fn, phi, alpha, beta=0.9, gamma=0.99, eps=1e-8, n_steps=100):
    # TODO  (use t+1 in the bias-correction exponents, t = 0, 1, 2, ...)
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 6.4
def _t64():
    gf = lambda p: np.array([0.1 * p[0], 10 * p[1]]) + np.array([np.cos(p[1]), 0.0])
    p0 = np.array([3.0, 1.0])
    # reference trajectories (computed independently)
    m = np.zeros(2); ph = p0.copy(); R1 = [ph.copy()]
    for _ in range(50): m = .9 * m + .1 * gf(ph); ph = ph - .05 * m; R1.append(ph.copy())
    m = np.zeros(2); ph = p0.copy(); R2 = [ph.copy()]
    for _ in range(50): m = .9 * m + .1 * gf(ph - .05 * .9 * m); ph = ph - .05 * m; R2.append(ph.copy())
    m = np.zeros(2); v = np.zeros(2); ph = p0.copy(); R3 = [ph.copy()]
    for t in range(50):
        g = gf(ph); m = .9 * m + .1 * g; v = .99 * v + .01 * g ** 2
        ph = ph - .05 * (m / (1 - .9 ** (t + 1))) / (np.sqrt(v / (1 - .99 ** (t + 1))) + 1e-8); R3.append(ph.copy())
    ok = [close(momentum(gf, p0, .05, .9, 50), R1, 1e-10), close(nesterov(gf, p0, .05, .9, 50), R2, 1e-10),
          close(adam(gf, p0, .05, .9, .99, 1e-8, 50), R3, 1e-10)]
    print("   momentum / nesterov / adam:", ["✅" if o else "❌" for o in ok])
    return all(ok)
check("6.4", 2, _t64, "Three optimisers in ~15 lines. torch.optim does exactly this.",
      "Path must include the starting point and have n_steps+1 rows.")

In [ ]:
#@title 🔑 Solution 6.4 — double-click to peek (have a go first!)
peek('6.4')
def momentum(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    phi = np.array(phi, float); m = np.zeros_like(phi); path = [phi.copy()]
    for _ in range(n_steps):
        m = beta * m + (1 - beta) * grad_fn(phi); phi = phi - alpha * m; path.append(phi.copy())
    return np.array(path)

def nesterov(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    phi = np.array(phi, float); m = np.zeros_like(phi); path = [phi.copy()]
    for _ in range(n_steps):
        m = beta * m + (1 - beta) * grad_fn(phi - alpha * beta * m); phi = phi - alpha * m; path.append(phi.copy())
    return np.array(path)

def adam(grad_fn, phi, alpha, beta=0.9, gamma=0.99, eps=1e-8, n_steps=100):
    phi = np.array(phi, float); m = np.zeros_like(phi); v = np.zeros_like(phi); path = [phi.copy()]
    for t in range(n_steps):
        g = grad_fn(phi); m = beta * m + (1 - beta) * g; v = gamma * v + (1 - gamma) * g ** 2
        mt = m / (1 - beta ** (t + 1)); vt = v / (1 - gamma ** (t + 1))
        phi = phi - alpha * mt / (np.sqrt(vt) + eps); path.append(phi.copy())
    return np.array(path)

In [ ]:
# 💻 Compare them on a long, narrow valley (Fig 6.9): steep in phi1, shallow in phi0
valley = lambda p: 0.5 * (0.05 * p[0] ** 2 + 5 * p[1] ** 2)
valley_grad = lambda p: np.array([0.05 * p[0], 5 * p[1]])
start = [-9.0, 1.5]
runs = {"GD (α=0.38)": gd(valley_grad, start, 0.38, 80), "momentum": momentum(valley_grad, start, 0.38, 0.9, 80),
        "Nesterov": nesterov(valley_grad, start, 0.38, 0.9, 80), "Adam": adam(valley_grad, start, 0.6, 0.9, 0.99, n_steps=80)}
V0, V1 = np.meshgrid(np.linspace(-10, 2, 200), np.linspace(-2, 2, 200))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].contour(V0, V1, valley([V0, V1]), 25, cmap="Greys", linewidths=0.6)
for name, p in runs.items():
    ax[0].plot(p[:, 0], p[:, 1], ".-", ms=3, lw=1, label=name)
    ax[1].semilogy([valley(q) + 1e-12 for q in p], label=name)
ax[0].legend(); ax[0].set_title("Paths"); ax[1].set_title("Loss vs step"); ax[1].legend(); plt.show()

### 🏆 Challenge 6 ⭐⭐ Race to the burrow
The **dune function** $L(a,b)=(1-a)^2+20\,(b-a^2)^2$ (a Rosenbrock cousin) has a curved, banana-shaped valley. The burrow is at $(1,1)$ and you start at $(-1.5,\,2)$.

Pick an optimiser and its hyperparameters. You win when the loss drops below $10^{-4}$, and your score is the **number of steps** it took (maximum 3,000). Fewer steps is better.
- 🥉 < 2000 steps
- 🥈 < 600 steps
- 🥇 < 200 steps

In [ ]:
dune = lambda p: (1 - p[0]) ** 2 + 20 * (p[1] - p[0] ** 2) ** 2
dune_grad = lambda p: np.array([-2 * (1 - p[0]) - 80 * p[0] * (p[1] - p[0] ** 2), 40 * (p[1] - p[0] ** 2)])

def race(path):
    L = np.array([dune(p) for p in path]); hit = np.flatnonzero(L < 1e-4)
    steps = int(hit[0]) if len(hit) else 10 ** 9
    D0, D1 = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-1, 3, 300))
    plt.figure(figsize=(5, 4)); plt.contour(D0, D1, np.log(dune([D0, D1]) + 1e-3), 30, cmap="YlOrBr")
    plt.plot(path[:, 0], path[:, 1], ".-", color=TB["green"], ms=2, lw=0.8); plt.plot(1, 1, "*", color=TB["red"], ms=15)
    plt.title(f"steps to burrow: {steps if steps < 10**9 else 'DNF'}"); plt.show()
    return steps

In [ ]:
my_path = gd(dune_grad, [-1.5, 2.0], lr=0.001, n=3000)   # TODO: swap in momentum / nesterov / adam and tune!

In [ ]:
# 🧪 Check puzzle C6
def _tc6():
    assert len(my_path) <= 3001 and np.allclose(my_path[0], [-1.5, 2.0]), "start at (-1.5, 2) and use <= 3000 steps"
    s = race(np.asarray(my_path)); print(f"   {s if s < 10**9 else 'did not finish'} steps → {medal(s, 2000, 600, 200)}")
    return s < 200
check("C6", 2, _tc6, "First into the burrow. The mob is impressed.",
      "Try adam(dune_grad, [-1.5, 2.0], alpha=0.05..0.3, ...) or momentum with beta ≈ 0.9 and a bigger lr.")

In [ ]:
#@title 🔑 Solution C6 — double-click to peek (have a go first!)
peek('C6')
my_path = adam(dune_grad, [-1.5, 2.0], alpha=0.2, beta=0.9, gamma=0.99, n_steps=3000)

---
# 🏁 Your progress

In [ ]:
progress()

### ⏭️ Next time
**Session 3 · Backprop and performance (Ch 7–8):** computing gradients efficiently, initialising deep nets, and whether any of it generalises.

*Built for the Transition Bio ML reading group · based on Prince, Understanding Deep Learning (MIT Press, CC-BY-NC-ND) · all data synthetic.*